# L10d Lab: Evaluation of Bandit Problem Algorithms
In this lab, we'll continue our discussion [of multi-armed bandit problems](https://en.wikipedia.org/wiki/Multi-armed_bandit) and look at the performance of the explore-first, $\epsilon$-greedy, and UCB1 algorithms on a simple stochastic bandit problem. The objective of a bandit algorithm is to discover the action (or "arm") that provides the highest expected reward. Then we implement Thompson sampling and use it to solve a binary Bernoulli bandit problem.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
>
> * __Run three bandit algorithms on the same problem:__ The explore-first, epsilon-greedy, and UCB1 algorithms each use the rewards seen so far to decide which arm to pull next. We'll run all three on a seven-arm bandit and count how often each one pulls each arm.
> * __Compare empirical regret with the theoretical bounds:__ Regret measures how much reward an algorithm gives up compared with always pulling the best arm. We'll compute the regret of each algorithm after every round and compare it with the regret bounds from the lecture.
> * __Implement Thompson sampling:__ Thompson sampling keeps a Beta distribution for the success probability of each arm and pulls the arm whose sample from that distribution is highest. We'll complete a Thompson sampling function and check that it learns which arm is best.

In [the L10c lecture](../L10c/CHEME-5800-L10c-Lecture-StochasticMultiarmBandits-Fall-2026.ipynb), we introduced the stochastic multi-armed bandit problem, regret, and algorithms that balance exploration and exploitation. Here, we run these algorithms on bandits where we know the answer, so we can see how each one finds the best arm.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package, this lab's code in the [src](src) folder, and the packages used here.

Let's set up our code environment:

In [ ]:
# Load packages and paths from this notebook's local setup file -
include(joinpath(@__DIR__, "Include.jl"));

# Choose one theme; after changing it, rerun this cell and the plotting cells -
theme(:default)   # light background
# theme(:dark)   # dark background

In addition to standard Julia libraries, we'll use [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl). Check out [the documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for more information on the functions, types, and data used in this material.

Your Thompson sampling function goes in [src/Compute.jl](src/Compute.jl). We call [the `L10dThompson.my_thompson_sampling(...)` function](src/Compute.jl) by its qualified name, so that re-running the setup cell picks up your edits.

### The Arms
First, let's implement the `world(...)` function. This function takes the action $a$ from our agent and returns the reward associated with choosing this action, i.e., $r\sim\mathcal{D}_{a}$. We'll use [a Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) to model the rewards, so every reward is a number between 0 and 1. A Beta distribution has two parameters $(\alpha,\beta)$ which determine its shape, and its expectation is $\alpha/(\alpha+\beta)$.

We store the reward distribution of each arm in the `arms::Array{Beta{Float64},1}` array; the bandit algorithms see only the rewards:

In [ ]:
arms = [
    Beta(2.0, 2.0),   # arm 1
    Beta(10.0, 2.0),  # arm 2
    Beta(2.0, 10.0),  # arm 3
    Beta(6.0, 20.0),  # arm 4
    Beta(20.0, 6.0),  # arm 5
    Beta(20.0, 10.0), # arm 6
    Beta(10.0, 20.0)  # arm 7
]; # the reward distribution 𝒟ₐ of each arm

function world(a::Int64)::Float64
    return rand(arms[a]); # one reward r ~ 𝒟ₐ, a number between 0 and 1
end;

Which arm has the highest expected reward? Only the world knows this answer, the bandit algorithms do not! Let's tabulate the parameters, mean reward, and standard deviation of each arm:

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm

    # compute the mean and the standard deviation of the reward of each arm -
    for a ∈ eachindex(arms)
        d = arms[a]; # the reward distribution of arm a
        (α, β) = params(d); # the two parameters of the Beta distribution
        push!(df, (action = a, alpha = α, beta = β,
            mean_reward = round(mean(d), digits = 3), sd_reward = round(std(d), digits = 3)));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ The winner is the arm with the highest expected reward, which is set by the parameters of its Beta distribution: arm 2, with a mean reward of $10/12\approx 0.833$.

Let's plot the reward distributions of arm 2 and another arm, arm 6. [The `pdf(...)` function](https://juliastats.org/Distributions.jl/stable/univariate/#Distributions.pdf-Tuple{UnivariateDistribution,%20Real}) evaluates the probability density of a distribution:

In [ ]:
let

    # initialize -
    x = range(0.0, 1.0, length = 501); # rewards between 0 and 1

    # plot the reward distributions of arm 2 and arm 6 -
    plot(x, pdf.(arms[2], x), lw = 2, label = "Arm 2: (α, β) = (10, 2)", framestyle = :box,
        xlabel = "Reward r (AU)", ylabel = "Probability density (AU)", legend = :topleft)
    plot!(x, pdf.(arms[6], x), lw = 2, label = "Arm 6: (α, β) = (20, 10)")
end

### Constants
Finally, let's set some constants we'll use in the subsequent tasks. See the comment beside the value for a description of what it is, its permissible values, etc.

In [ ]:
K = 7; # number of arms for the bandit
T = 5000; # number of rounds for each decision task
seed = 5800; # seed for the random number generator, so each game can be repeated

___

## Task 1: Run the Three Bandit Algorithms
In this task, we run the explore-first, $\epsilon$-greedy, and UCB1 algorithms on our `K`-arm bandit problem and see which arm each one pulls. We've created a custom type for each algorithm, which holds the number of arms `K`. We create an instance of the type using [the `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build), then pass it, the number of rounds `T`, and our `world(...)` function to [the `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.solve-Tuple{AbstractBanditAlgorithmModel}), which plays the game.

__Output__: [The `solve(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.solve-Tuple{AbstractBanditAlgorithmModel}) returns a results array $\mathbf{U}$ holding the records of the game. The rows of the array are the rounds of the decision task, while each column corresponds to the action $a\in\mathcal{A}$. If $u_{t,a}>0$, then action $a$ was chosen in round $t$, and $u_{t,a}$ is the reward received for taking action $a$.

The algorithms and our `world(...)` function draw random numbers, so each game cell first sets the seed with [the `Random.seed!(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.seed!) and gives the same game every time we run it.

__Prediction:__ After 5000 rounds, which algorithm will have pulled the best arm, arm 2, most often: {explore-first, $\epsilon$-greedy, or UCB1}?

### Explore-First Algorithm
The explore-first algorithm pulls each arm $N_{a}$ times, then pulls the arm with the highest average reward for the remaining rounds. The lecture writes the number of exploration pulls with a constant hidden in the $\mathcal{O}(\cdot)$ notation, $N_{a} = (T/K)^{2/3}\cdot\mathcal{O}\left(\log{T}\right)^{1/3}$. The package takes that constant to be one and rounds to the nearest integer:
$$
N_{a} = \text{round}\left((T/K)^{2/3}\cdot\left(\log{T}\right)^{1/3}\right)
$$

In [ ]:
Nₐ = ((T/K)^(2/3))*(log(T))^(1/3) |> x -> round(Int, x); # number of times we pull each arm while exploring
println("each arm is pulled Nₐ = ", Nₐ, " times, so exploring takes the first ", K*Nₐ, " rounds")

We create [an instance of the `MyExploreFirstAlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyExploreFirstAlgorithmModel) and play the game. We store the results array in the `results_explore_first::Array{Float64,2}` variable:

In [ ]:
results_explore_first = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyExploreFirstAlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    results = solve(m, T = T, world = world); # T rounds against the world

    results # return
end;

### Epsilon-Greedy Algorithm
In each round $t$, the $\epsilon$-greedy algorithm pulls a random arm with probability $\epsilon_{t}$, and otherwise pulls the arm with the highest average reward so far. The package lets the exploration probability shrink as the game goes on:
$$
\epsilon_{t} = t^{-1/3}\cdot\left(\log(K\cdot{t})\right)^{1/3}
$$
This is a slightly different choice from the lecture's $\epsilon_{t}=t^{-1/3}\cdot\left(K\cdot\log{t}\right)^{1/3}$; both shrink like $t^{-1/3}$. In the first three rounds, $\epsilon_{t}>1$, so every pull is random.

We create [an instance of the `MyEpsilonGreedyAlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyEpsilonGreedyAlgorithmModel) and play the game. We store the results array in the `results_eps_greedy::Array{Float64,2}` variable:

In [ ]:
results_eps_greedy = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyEpsilonGreedyAlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    results = solve(m, T = T, world = world); # T rounds against the world

    results # return
end;

### UCB1 Algorithm
The UCB1 algorithm pulls each arm once. Then, in each round $t$, it pulls the arm with the highest upper confidence bound, the estimated mean reward $\bar{\mu}(a)$ plus a bonus:
$$
\bar{\mu}(a)+\sqrt{\frac{2\log(t)}{N_{a}}}
$$
where $N_{a}$ is the number of times arm $a$ has been pulled so far. The bonus is large for an arm we have pulled only a few times, so the algorithm is optimistic about the arms it knows least about.

We create [an instance of the `MyUCB1AlgorithmModel` type](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/bandit/#VLDataScienceMachineLearningPackage.MyUCB1AlgorithmModel) and play the game. We store the results array in the `results_UCB1::Array{Float64,2}` variable:

In [ ]:
results_UCB1 = let

    # initialize -
    Random.seed!(seed); # set the seed, so this cell gives the same game every time
    m = build(MyUCB1AlgorithmModel, (
        K = K, # number of arms
    ));

    # play the game -
    results = solve(m, T = T, world = world); # T rounds against the world

    results # return
end;

Let's look at the first ten rounds of the UCB1 game:

In [ ]:
results_UCB1[1:10, :] # rounds 1 to 10, all arms

__What do we see?__ Each row has one nonzero entry, the reward for the arm pulled in that round. In the first seven rounds, UCB1 pulls each arm once.

Now let's count how many times each algorithm pulled each arm, i.e., the number of rounds with $u_{t,a}>0$:

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm

    # count the rounds in which each algorithm pulled each arm -
    for a ∈ 1:K
        push!(df, (
            arm = a,
            mean_reward = round(mean(arms[a]), digits = 3), # known only to us, the oracle
            explore_first = count(results_explore_first[:, a] .> 0), # rounds with u_(t,a) > 0
            eps_greedy = count(results_eps_greedy[:, a] .> 0),
            UCB1 = count(results_UCB1[:, a] .> 0),
        ));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ With our seed, all three algorithms pulled arm 2 most often. Explore-first pulled every arm 163 times, then arm 2 for the rest of the game. $\epsilon$-greedy pulled arm 2 4,223 times and spread its random pulls about evenly over the other arms. UCB1 pulled arm 2 only 3,518 times, and arm 5 946 times. Your counts may differ if you change the seed.

### Things to think about

__Question:__ Explore-first and $\epsilon$-greedy pulled the six other arms about equally often, while UCB1 pulled arm 5 almost a thousand times and arm 3 only 32 times. Why does UCB1 spend its exploration on arm 5? Briefly explain.

___

## Task 2: Compute the Empirical Regret for Each Algorithm
In this task, we compute the empirical regret of each algorithm and see which algorithm gives the best actual performance, and which was predicted to have the best theoretical performance.

__Regret__: Let $\mu^{\star}$ be the mean of the best arm, i.e., $\mu^{\star} = \max_{a\in\mathcal{A}}\mu(a)$. The regret $R(t)$ of an algorithm after $t$ rounds is defined as:
$$
\begin{align*}
R(t) = t\cdot\mu^{\star} - \sum_{s=1}^{t}r_{s}
\end{align*}
$$
The first term is the expected reward from always choosing the best arm over the $t$ rounds. The second term is the total reward obtained by the agent over the $t$ rounds, where $r_{s}$ is the reward received at round $s$ for the action chosen by the agent.

We are the oracle, so we know the mean reward of every arm: $\mu^{\star}=10/12\approx 0.833$, the mean of arm 2. For each algorithm, we compute $R(t)$ after every round $t=1,2,\dots,T$ and store the curves in the `regret_curves::Dict{String, Array{Float64,1}}` dictionary:

In [ ]:
regret_curves = let

    # initialize -
    μstar = maximum(mean.(arms)); # μ⋆, the mean reward of the best arm: known only to us, the oracle
    games = [("Explore-first", results_explore_first), ("ε-greedy", results_eps_greedy), ("UCB1", results_UCB1)]; # (name, results array)
    regret_curves = Dict{String, Array{Float64,1}}(); # the regret after each round, for each algorithm

    # main loop -
    for (name, U) ∈ games
        r = vec(sum(U, dims = 2)); # the reward in each round: each row of U has one nonzero entry
        R = zeros(T); # the regret after each round
        total = 0.0; # the reward collected so far
        for t ∈ 1:T
            total = total + r[t]; # r₁ + r₂ + ... + rₜ
            R[t] = t*μstar - total; # R(t) = tμ⋆ - (r₁ + r₂ + ... + rₜ)
        end
        regret_curves[name] = R; # store the curve for this algorithm
    end

    regret_curves # return
end;

Let's plot the regret curves.

__Prediction:__ Which algorithm will have the smallest regret after 5000 rounds: {explore-first, $\epsilon$-greedy, or UCB1}?

In [ ]:
let

    # initialize -
    labels = ["Explore-first", "ε-greedy", "UCB1"]; # plot the curves in this order
    p = plot(framestyle = :box, xlabel = "Round t", ylabel = "Regret R(t) (AU)", legend = :topleft);

    # plot the regret after each round -
    for name ∈ labels
        plot!(p, 1:T, regret_curves[name], lw = 2, label = name);
    end

    p # return
end

__What do we see?__ Explore-first's regret climbs for the first 1,141 rounds, while it pulls every arm in turn, then levels off once it pulls only arm 2. The $\epsilon$-greedy and UCB1 curves stay close for about 1,500 rounds. After that, $\epsilon$-greedy's regret keeps climbing, because it keeps pulling random arms, while UCB1's grows more and more slowly. After 5000 rounds, UCB1 has the smallest regret, about 232, compared with about 308 for $\epsilon$-greedy and about 400 for explore-first.

### Theoretical regret bounds
Finally, let's compare the empirical regret with the theoretical regret bounds from the lecture. The bounds hide unknown constants in the $\mathcal{O}(\cdot)$ notation. We set those constants to one, so the numbers below show how each bound scales, not a proven limit:

* __Explore-first__ and __$\epsilon$-greedy__: The expected regret over $T$ rounds is at most a constant times ${T}^{2/3}\cdot\left(K\cdot\log{T}\right)^{1/3}$, which grows like $T^{2/3}$.
* __UCB1__: The expected regret over $T$ rounds is at most a constant times $\sqrt{K\cdot{T}\cdot\log{T}}$, which grows like $\sqrt{T}$, apart from the logarithm, the best asymptotic performance of the three.

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per algorithm
    bound_explore = (T^(2/3))*(K*log(T))^(1/3); # explore-first and ε-greedy bound, T^(2/3)⋅(K⋅log T)^(1/3)
    bound_UCB1 = sqrt(K*T*log(T)); # UCB1 bound, √(K⋅T⋅log T)

    # compare the regret after T rounds with the bound -
    for (name, bound) ∈ [("Explore-first", bound_explore), ("ε-greedy", bound_explore), ("UCB1", bound_UCB1)]
        push!(df, (algorithm = name, regret = round(regret_curves[name][end], digits = 1), bound_scale = round(bound, digits = 1)));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ All three regrets are well below these scales. The bounds cover every bandit with rewards between 0 and 1 and describe the expected regret, so for one run on one particular bandit they can be far from the regret we see. However, they also tell us how fast regret grows with the number of rounds.

__Prediction:__ Suppose we double the number of rounds to $T=10{,}000$. Will each algorithm's regret {double, grow by less than double, or stay about the same}?

In [ ]:
let

    # initialize -
    T_long = 10_000; # twice as many rounds
    μstar = maximum(mean.(arms)); # μ⋆, the mean reward of the best arm
    models = [("Explore-first", MyExploreFirstAlgorithmModel), ("ε-greedy", MyEpsilonGreedyAlgorithmModel), ("UCB1", MyUCB1AlgorithmModel)]; # (name, algorithm type)

    # play each game for T_long rounds, and compare its regret with the regret after T rounds -
    for (name, M) ∈ models
        Random.seed!(seed); # the same seed as before
        U = solve(build(M, (K = K,)), T = T_long, world = world); # T_long rounds against the world
        R = T_long*μstar - sum(U); # R(T_long): sum(U) adds up every reward the agent received
        println(name, ": R(", T, ") = ", round(regret_curves[name][end], digits = 1), ", R(", T_long, ") = ",
            round(R, digits = 1), ", ratio = ", round(R/regret_curves[name][end], digits = 2))
    end
end

__What do we see?__ Doubling the number of rounds made the regret of explore-first and $\epsilon$-greedy grow by about 60 percent, and the regret of UCB1 by only about a quarter. For comparison, doubling $T$ multiplies the first bound by about 1.63 and the UCB1 bound by about 1.47.

### Things to think about

__Question:__ UCB1 pulled the best arm less often than $\epsilon$-greedy (3,518 times versus 4,223), yet it has the smallest regret. How can that be? Why did its regret grow the least when we doubled the number of rounds? Briefly explain.

___

## Task 3: Implement Thompson Sampling
In this task, we implement Thompson sampling and use it to solve a binary Bernoulli bandit problem, where every reward is a success, $r=1$, or a failure, $r=0$.

Suppose each choice $a\in\mathcal{A}$ succeeds with a probability $\theta_{a}$ that the decision agent _does not know_. However, we are [the Oracle](https://en.wikipedia.org/wiki/The_Oracle_(The_Matrix)), so we know the actual probabilities. By experimenting in the world, can the agent learn which choice is most likely to succeed? Let's set up a world with three choices, with success probabilities $\theta_{1}=0.90$, $\theta_{2}=0.60$, and $\theta_{3}=0.80$. The `bernoulli_world(...)` function returns a reward of `1` with probability $\theta_{a}$ and `0` otherwise, using [a Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution):

In [ ]:
θ = [0.90, 0.60, 0.80]; # the probability of success of each choice: only the world knows these

function bernoulli_world(a::Int64, θ::Array{Float64,1}, rng::AbstractRNG)::Int64
    return rand(rng, Bernoulli(θ[a])) ? 1 : 0; # true (a success) with probability θ[a]
end;

### Implement Thompson Sampling
The agent models its uncertainty about each $\theta_{a}$ with a Beta distribution. Before any pulls, its belief about arm $a$, the __prior__, is $\text{Beta}(\alpha(a),\beta(a))$; we use $\alpha(a)=\beta(a)=1$, which is uniform on $[0,1]$. After $S(a)$ successes and $F(a)$ failures on arm $a$, its updated belief, the __posterior__, is $\text{Beta}(\alpha(a)+S(a),\beta(a)+F(a))$.

Let's write Thompson sampling ourselves, following the L10c algorithm. Open [`src/Compute.jl`](src/Compute.jl) and complete [the `my_thompson_sampling(...)` function](src/Compute.jl). Everything except four lines is already written, and a TODO comment marks each step:

> __Implementation tasks__
>
> * __TODO 1:__ Sample from the posterior of each arm. Replace `0.0` in `p[a] = 0.0` with one sample from $\text{Beta}(\alpha(a)+S(a),\beta(a)+F(a))$, drawn with the random number generator `rng`.
> * __TODO 2:__ Choose the arm with the highest sample. Replace `1` in `aₜ = 1` with the arm $a$ that maximizes the sample $\mathbf{p}(a)$.
> * __TODO 3:__ Update the counts of the arm we pulled. Replace the right-hand sides of the two lines so that $S(a_{t})\getsS(a_{t})+r_{t}$ and $F(a_{t})\getsF(a_{t})+(1-r_{t})$.

Save the file and re-run the setup cell at the top of the notebook to reload your edits, then run the cell below. Until your function is complete, the call throws an error.

We play $T=5000$ rounds, with one random number generator for the agent and the world, created from our `seed` by [the `MersenneTwister(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.MersenneTwister). We save the returned `NamedTuple` in the `result` variable. Its fields are the arm pulled and the reward received in each round, `actions` and `rewards`, and the successes and failures on each arm, `S` and `F`:

In [ ]:
result = let

    # initialize -
    rng = MersenneTwister(seed); # one random number generator for the agent and the world
    world_function(a) = bernoulli_world(a, θ, rng); # the agent sees only the reward of the arm it pulls

    # play T rounds with Thompson sampling -
    L10dThompson.my_thompson_sampling(world_function, length(θ), T, rng = rng) # the untouched stub throws an error
end;
println("pulls of each arm: ", result.S .+ result.F)

### Check: Your Thompson Sampling

Every round adds one success or one failure to the arm we pulled, so the counts must agree with the actions and the rewards, and the same seed must give the same game. We also play a test bandit whose best arm is the last one, $\theta=(0.70, 0.50, 0.90)$, which Thompson sampling should pull in most rounds:

In [ ]:
@testset "My Thompson sampling" begin

    # every round adds one success or one failure to the arm we pulled -
    @test sum(result.S) + sum(result.F) == T
    @test result.S .+ result.F == [count(result.actions .== a) for a ∈ eachindex(θ)] # pulls of each arm
    @test result.S == [sum(result.rewards[result.actions .== a]) for a ∈ eachindex(θ)] # successes on each arm

    # the same seed gives the same game, every time we play it -
    for k ∈ 1:2
        rng = MersenneTwister(seed); # a new random number generator with the same seed
        again = L10dThompson.my_thompson_sampling(a -> bernoulli_world(a, θ, rng), length(θ), T, rng = rng);
        @test again.actions == result.actions
    end

    # a test bandit whose best arm is the last one -
    θ_test = [0.70, 0.50, 0.90]; # success probabilities of the test bandit
    rng_test = MersenneTwister(1234); # its own random number generator
    test = L10dThompson.my_thompson_sampling(a -> bernoulli_world(a, θ_test, rng_test), 3, 2000, rng = rng_test);
    @test count(test.actions .== 3) > 1000 # the best arm is pulled in most of the 2000 rounds
    @test abs((1 + test.S[3])/(2 + test.S[3] + test.F[3]) - 0.90) < 0.05 # its posterior mean is close to 0.9
end;

Let's compare what the agent believes with the ground truth that only we know. The agent's estimate of $\theta_{a}$ is the posterior mean:
$$
\hat{\theta}_{a} = \frac{\alpha(a)+S(a)}{\alpha(a)+\beta(a)+S(a)+F(a)}
$$

__Prediction:__ For which arm will the estimate $\hat{\theta}_{a}$ be closest to the true value: {arm 1, arm 2, or arm 3}?

In [ ]:
let

    # initialize -
    df = DataFrame(); # one row per arm
    α = 1.0; # prior parameter α(a), the same for every arm
    β = 1.0; # prior parameter β(a), the same for every arm

    # compare the posterior mean with the true success probability -
    for a ∈ eachindex(θ)
        S = result.S[a]; # successes on arm a
        F = result.F[a]; # failures on arm a
        push!(df, (arm = a, θ = θ[a], pulls = S + F, S = S, F = F, θ̂ = round((α + S)/(α + β + S + F), digits = 3)));
    end

    # show the table -
    pretty_table(df;
        show_first_column_label_only = true,
        display_size = (-1, -1), # show every row and column
        alignment = :r # right-align every column
    );
end

__What do we see?__ With our seed, the agent pulled arm 1 in 4,908 of the 5000 rounds, and its estimate of $\theta_{1}$, 0.905, is close to the true 0.90. It pulled arm 3 only 81 times and arm 2 only 11 times, so its estimates for those arms, 0.819 and 0.538, are much rougher.

Let's plot the posterior of each arm with [the `pdf(...)` function](https://juliastats.org/Distributions.jl/stable/univariate/#Distributions.pdf-Tuple{UnivariateDistribution,%20Real}):

In [ ]:
let

    # initialize -
    x = range(0.0, 1.0, length = 2001); # success probabilities between 0 and 1, on a fine grid: the posterior of arm 1 is narrow
    p = plot(framestyle = :box, xlabel = "Probability of success (AU)", ylabel = "Probability density (AU)", legend = :topleft);

    # plot the posterior of each arm -
    for a ∈ eachindex(θ)
        d = Beta(1.0 + result.S[a], 1.0 + result.F[a]); # the posterior of arm a, with α(a) = β(a) = 1
        plot!(p, x, pdf.(d, x), lw = 2, label = "Arm $(a): θ = $(θ[a])");
    end

    p # return
end

__What do we see?__ The posterior of arm 1 is a tall, narrow peak near 0.9: after thousands of pulls, the agent is nearly sure of $\theta_{1}$. The posterior of arm 3 is wider, and the posterior of arm 2 is broad.

__Prediction:__ Suppose arm 3 is almost as good as arm 1, with $\theta_{3}=0.88$ instead of 0.80. Will Thompson sampling pull arm 3 {more often, less often, or about as often} as before?

In [ ]:
let

    # initialize -
    θ_close = [0.90, 0.60, 0.88]; # arm 3 is now almost as good as arm 1
    rng = MersenneTwister(seed); # the same seed as before
    world_function(a) = bernoulli_world(a, θ_close, rng); # the agent sees only the reward of the arm it pulls

    # play T rounds with Thompson sampling -
    close_result = L10dThompson.my_thompson_sampling(world_function, length(θ_close), T, rng = rng);
    println("θ = ", θ_close, ": pulls of each arm ", close_result.S .+ close_result.F)
    println("posterior means: ", round.((1 .+ close_result.S) ./ (2 .+ close_result.S .+ close_result.F), digits = 3))
end

__What do we see?__ With $\theta_{3}=0.88$, the agent pulled arm 3 551 times instead of 81. It takes more pulls to tell two nearly equal arms apart.

### Things to think about

__Question:__ After 5000 rounds, the agent's estimate of $\theta_{2}$ is still rough, because it pulled arm 2 only 11 times. Is that a problem for the agent? Why does Thompson sampling pull an arm so rarely, even before it knows that arm's success probability well? How is this like what UCB1 did with arm 3 in Task 1? Briefly explain.

___

## Summary
In this lab, we ran the explore-first, $\epsilon$-greedy, and UCB1 algorithms on a seven-arm bandit, compared their regret with the theoretical bounds, and implemented Thompson sampling for a binary Bernoulli bandit.

> __Key Takeaways:__
>
> * __Three ways to balance exploration and exploitation:__ All three algorithms found the best arm, but they explored in different ways. Explore-first and epsilon-greedy spread their exploration evenly over the arms, while UCB1 spent most of its exploration on the arms whose mean rewards were close to the best.
>
> * __Regret depends on which arms we explore:__ UCB1 pulled the best arm less often than epsilon-greedy but had the smallest regret, because most of its other pulls went to arms that were almost as good. When we doubled the number of rounds, its regret grew the least, as the theoretical bounds suggest.
>
> * __Thompson sampling keeps a probability model of each arm:__ We completed a Thompson sampling function that samples from the posterior of each arm and updates the success and failure counts after each pull. It pulled the best arm in almost every round, and it pulled the worse arms only rarely, even before it knew their success probabilities well.

We can now change the arms, the number of rounds, or the seed and see how the pulls and the regret of each algorithm respond.

___